In [1]:
!pip install langgraph langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.1 MB/s eta 0:00:00


In [2]:
from langchain_groq import ChatGroq
llm = ChatGroq(api_key="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", model="llama-3.1-8b-instant", temperature=0)

In [4]:
from typing import TypedDict

class TeamState(TypedDict):
    task: str # the original problem
    worker_result: str # what the worker computed
    summary: str # the supervisor's final summary

In [6]:
def worker(state: TeamState) -> dict:
    answer = llm.invoke('Solve this math problem, show the number only: ' + state['task']).content
    return {'worker_result': answer}

def supervisor(state: TeamState) -> dict:
    summary = llm.invoke(
        f"The worker solved '{state['task']}' and got "
        f"{state['worker_result']}. Write a one-line summary."
    ).content
    return {'summary': summary}

In [8]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(TeamState)
builder.add_node('worker', worker)
builder.add_node('supervisor', supervisor)
builder.add_edge(START, 'worker')
builder.add_edge('worker', 'supervisor')
builder.add_edge('supervisor', END)
graph = builder.compile()

In [10]:
result = graph.invoke({'task': 'What is 144 divided by 12, then plus 5?'})
print('Worker result:', result['worker_result'])
print('Supervisor summary:', result['summary'])

Worker result: 12
Supervisor summary: The worker incorrectly solved the math problem 'What is 144 divided by 12, then plus 5?' and arrived at an answer of 12, which is incorrect.
